# Random forest vs. hedonic OLS: art auction prices

APMA 4903 project. This notebook runs the random forest part of our analysis:

1. Fit the hedonic model by OLS on 21,783 training sales (Nov 2023 – Dec 2025).
2. Tune and fit a random forest (500 trees) on the same sales and the same seven characteristics.
3. Compare both on 8,458 sales from 2026 that neither model has seen.

**How to run:** choose **Runtime → Run all**. Nothing to upload: the first cell downloads the project from GitHub. The whole notebook takes about 5 minutes.

## 1. Get the code and data

In [ ]:
import os
# Download the project from GitHub (skipped if this notebook is already inside it)
if not os.path.exists('src/random_forest.py'):
    if not os.path.exists('apma-project'):
        !git clone -q https://github.com/ajavakhishvili16/apma-project.git
    os.chdir('apma-project/random_forest')
print('Working in', os.getcwd())
print('Training data found:', os.path.exists('../training_set_2023_2025_clean-2.csv'))
print('Test data found:    ', os.path.exists('../test_set_2026_clean-2.csv'))

## 2. Hedonic model (OLS)

ln(P/A) on exposure, fame, ln A, (ln A)², medium, support, sale season and creation period; standard errors clustered by artist.

In [ ]:
!python src/hedonic_ols.py

## 3. Random forest: tune, fit, and compare on 2026 sales

Takes about a minute. Settings are tuned on training data only (fit to June 2025, validate on July–December 2025).

In [ ]:
!python src/random_forest.py

## 4. Results table

In [ ]:
import json, pandas as pd
r = json.load(open('results/rf_vs_ols_metrics.json'))
t = pd.DataFrame(r['test']).T.rename(index={'predict_mean':'Baseline: training mean','hedonic_ols':'Hedonic OLS','random_forest':'Random forest'})
t.columns = ['R²','RMSE (log)','MAE (log)','Median error factor','Share within 1.5×']
display(t.round(3))
lo, hi = r['R2_gain_95ci_bootstrap']
print(f"Forest R² gain: {r['R2_gain']:.3f}  (95% bootstrap interval {lo:.3f} to {hi:.3f})")
print(f"Training R²: OLS {r['train']['hedonic_ols']['R2']:.3f}, forest {r['train']['random_forest']['R2']:.3f}")
print('Forest settings:', r['forest_settings'])

## 5. What the forest learned: size and fame

In [ ]:
from IPython.display import Image
Image('results/rf_partial_dependence.png')

## 6. Validation: how B, m and the leaf size were chosen

Uses only the training sales: fit on sales before July 2025, score on July–December 2025.
Left: validation R² against the minimum leaf size, one line per m. Right: validation R² as trees are added.

In [ ]:
!python src/validation.py
from IPython.display import Image
Image('results/validation_curves.png')

## 7. Averaging trees on the 2026 test set

The same averaging effect on the held-out 2026 sales (Part 4, aggregation).

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, 'src')
from sklearn.ensemble import RandomForestRegressor
from common import BASE, TRAIN, TEST, load, metrics
from random_forest import forest_inputs
%matplotlib inline
train = load(TRAIN)
levels = {c: sorted(train[c].astype(str).unique()) for c in BASE}
test = load(TEST)
Xtr, Xte = forest_inputs(train, levels), forest_inputs(test, levels)
st = r['forest_settings']   # the settings chosen by validation in section 3
rf = RandomForestRegressor(st['trees'], min_samples_leaf=st['min_samples_leaf'], max_features=st['max_features'], n_jobs=-1, random_state=0).fit(Xtr, train.y)
tree_preds = np.array([t.predict(Xte.values) for t in rf.estimators_])
running = np.cumsum(tree_preds, axis=0) / np.arange(1, 501)[:, None]
r2_by_B = [metrics(test.y, running[b])['R2'] for b in range(500)]
ols_r2 = r['test']['hedonic_ols']['R2']

plt.figure(figsize=(9, 4.5))
plt.plot(range(1, 501), r2_by_B, color='#0F6E56', lw=2.5, label='Random forest')
plt.axhline(ols_r2, ls='--', color='#444441', label=f'Hedonic OLS ({ols_r2:.3f})')
plt.xscale('log'); plt.xlabel('Number of trees averaged (B)'); plt.ylabel('R² on 2026 sales')
plt.title('Averaging trees'); plt.legend(frameon=False); plt.show()
for B in [1, 10, 100, 500]:
    print(f'B = {B:>3}: R² {r2_by_B[B-1]:.3f}')